In [ ]:
%pip install -Uqq fastai fastdownload duckduckgo-search ipywidgets

In [ ]:
# Import required modules
from fastdownload import FastDownload, download_url
from fastai.vision.all import *
from fastai.vision.widgets import ImageClassifierCleaner
from duckduckgo_search import DDGS
from urllib.error import HTTPError, URLError
from PIL import UnidentifiedImageError
from requests.exceptions import RequestException

# Use the duckduckgo search function
with DDGS() as ddgs:
    urls = ddgs.images("cat", max_results=5)
    for i, r in enumerate(urls):
        try:
            dest = f'cat{i}.jpg'  # Destination file name
            image_url = r['image']  # Image URL

            response = requests.get(image_url, allow_redirects=True)  # Get the image response
            if response.status_code == 403:
                continue
            
            download_url(r['image'], dest, show_progress = False) # Download the image using fastdownload
            print(r['image'], r['title']) # Print the image URL and title

            print(r['title']) # Print the title of the image
            print('-' * 40) # Print a separator line
            print()
            display(Image.open(dest).to_thumb(256, 256))   # Display the image using fastai's display function

        except Exception as e:
            print(f"An error occurred: {e}")
            continue

In [ ]:
searches = 'cat', 'dog', 'fox' # Search terms for images
path = Path('animal_predict') # Path to save the images

for o in searches: # Loop through each search term
    dest = (path/o) # Destination path for the downloaded images
    dest.mkdir(parents=True, exist_ok=True) # Create the directory if it doesn't exist

    with DDGS() as ddgs: 
        res = ddgs.images(f'{o} photo', max_results=100) # Search for images using duckduckgo
        urls = [r['image'] for r in res] # Get the image URLs

    for i, url in enumerate(urls):
        try:
            download_url(url, dest/f"{i}.jpg", show_progress=False)
        except Exception as e:
            continue # Skip any exceptions that occur during download
        
    time.sleep(1) # Sleep for 1 second to avoid overwhelming the server
    resize_images(path, max_size=400, dest = path) # Resize the images to a maximum size of 400 pixels

failed = verify_images(get_image_files(path)) # Verify the images to check for any failed downloads
failed.map(Path.unlink) # Delete the failed images
print(len(failed), 'images failed to download.') # Print the number of images deleted

dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), # Define the data block with image and category blocks
    get_items=get_image_files, # Get the image files
    splitter=RandomSplitter(valid_pct=0.2, seed=42), # Split the data into training and validation sets
    get_y=parent_label, # Get the labels from the parent folder names
    item_tfms=[Resize(192, method='squish')] # Resizes and reshapes the images
).dataloaders(path, bs=32) # Create the data loaders with a batch size of 32

dls.show_batch(max_n=6) # Show a batch of images with their labels

In [ ]:
tempsearch = 'fox'
tempdest = f'{tempsearch}.jpg' # Temporary destination for the image

with DDGS() as ddgs:
    urls = ddgs.images(f"{tempsearch} photo", max_results=5) # Search for images using duckduckgo, multiple images searched in case of broken links

for u in urls: # Loop through the image URLs till a successful download
    try:
        url = u['image'] # Get the image URL
        download_url(url, tempdest, show_progress=False) # Download the image using fastdownload
        break # Break the loop after the first successful download
    except Exception as e:
        print(f"Skipping broken URL")

if not os.path.exists(tempdest): # Check if the file exists
    raise FileNotFoundError(f"File {tempdest} not found.")

display(Image.open(tempdest).to_thumb(256, 256)) # Display the downloaded image

learn = vision_learner(dls, resnet18, metrics=error_rate) # Create a vision learner using the data loaders and a pre-trained ResNet18 model
learn.fine_tune(3) # Fine-tune the model for 3 epoch

guess_animal,_,probs = learn.predict(PILImage.create(tempdest)) # Make a prediction on the downloaded image
print(f"Prediction: This is a {guess_animal}, Probability that it is a {searches[2]}: {probs[2]:.4f}") # probs[>>INSERT INDEX HERE FOR CAT, DOG, FOX<<] ~~ Print the prediction and probability. 

In [ ]:
animals = DataBlock( # Create a new DataBlock for the animal images
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(seed=42),
    get_y=parent_label,
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms()
)

dls = animals.dataloaders(path) # Create the data loaders with the new transformations

learn = vision_learner(dls, resnet18, metrics=error_rate) # Create a new vision learner with the updated data loaders and a pre-trained ResNet18 model
learn.fine_tune(4) # Fine-tune the model for 4 epochs

interp = ClassificationInterpretation.from_learner(learn) # Create an interpretation object from the learner
interp.plot_confusion_matrix() # Plot the confusion matrix to visualize the model's performance

interp.plot_top_losses(5, nrows=1) # Plot the top losses to visualize the worst-performing images

In [ ]:
cleaner = ImageClassifierCleaner(learn) # Create an image classifier cleaner object
cleaner

In [ ]:
for idx in cleaner.delete(): # Loop through the indices of the images to delete
    os.remove(cleaner.fns[idx]) # Remove the images from the file system

for i in searches:
    for idx,i in cleaner.change(): # Loop through the indices of the images to change
        shutil.move(cleaner.fns[idx], path/i) # Move the images to the new category folder